In [0]:
dbutils.widgets.text("catalog", "tesco_bank_training", "Catalog")
dbutils.widgets.text("schema", "datasets", "Schema")
dbutils.widgets.text("write_schema", "", "write_schema ( Your name ie. jack_gibb)")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
write_schema = dbutils.widgets.get("schema")

customers_table = f"{catalog}.{schema}.customers"
repayments_table = f"{catalog}.{schema}.repayments"
transactions_table = f"{catalog}.{schema}.transactions"

write_schema = dbutils.widgets.get("write_schema")

write_schema = write_schema + "_" + "silver"

print(customers_table)
print(repayments_table)
print(transactions_table)
print(write_schema)

tesco_bank_training.datasets.customers
tesco_bank_training.datasets.repayments
tesco_bank_training.datasets.transactions
jack_gibb_silver


## 1. Read data into a Spark DataFrame

In [0]:
# Read table into dataframe using spark.table() 
customers = spark.table(customers_table)

## 2. Write data into a Unity table

In [0]:
# Writing - overwrite mode
customers.write.mode("overwrite").saveAsTable(f"{catalog}.{write_schema}.customer_version_history")

In [0]:
# Writing - append mode (this step is our "mistake" that we want to reverse)
customers.write.mode("append").saveAsTable(f"{catalog}.{write_schema}.customer_version_history")

## 3. The Describe and Restore Command

In [0]:
# Running a describe history on the table 
display(spark.sql(f"""DESCRIBE HISTORY {catalog}.{write_schema}.customer_version_history"""))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-08-07T10:28:14.000Z,76835418810984,jack.gibb@inov8consulting.co.uk,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(431825730178474),c4deedfb-fb74-4993-9ad9-96530e48cd55,0807-080044-ic7c55am-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 815, numOutputBytes -> 21079)",null,Databricks-Runtime/18.x-photon-scala2.13
0,2026-08-07T10:28:12.000Z,76835418810984,jack.gibb@inov8consulting.co.uk,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(431825730178474),e267cde7-5b7e-49cd-a510-46c6ec46f876,0807-080044-ic7c55am-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 815, numOutputBytes -> 20464)",null,Databricks-Runtime/18.x-photon-scala2.13


In [0]:
# This restore command will restore the table to the first version of its history before the append
spark.sql(f"""RESTORE TABLE {catalog}.{write_schema}.customer_version_history TO VERSION AS OF 0""")

DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]

## 4. GDPR Delete Request

In [0]:
# Deleting the customer from the table
spark.sql(f"""DELETE FROM {catalog}.{write_schema}.customer_version_history WHERE customer_id = 'CUST00377' """)

DataFrame[num_affected_rows: bigint]

In [0]:
# Checking the record has been deleted from the table
display(spark.sql(f"""SELECT * FROM {catalog}.{write_schema}.customer_version_history WHERE customer_id = 'CUST00377' """))

customer_id,first_name,last_name,date_of_birth,city,employment_status,annual_income,product_type,account_opened,credit_limit


In [0]:
# Running a describe history on the table to see the version we are looking for
display(spark.sql(f"""DESCRIBE HISTORY {catalog}.{write_schema}.customer_version_history"""))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-08-07T10:28:23.000Z,76835418810984,jack.gibb@inov8consulting.co.uk,DELETE,"Map(predicate -> [""(customer_id#16915 = CUST00377)""])",null,List(431825730178474),163eb31f-25a1-4455-8825-dca1eccbbcfe,0807-080044-ic7c55am-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1106, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 756, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 350)",null,Databricks-Runtime/18.x-photon-scala2.13
2,2026-08-07T10:28:19.000Z,76835418810984,jack.gibb@inov8consulting.co.uk,RESTORE,"Map(version -> 0, timestamp -> null)",null,List(431825730178474),759665a4-a861-4004-aa42-d5965e332127,0807-080044-ic7c55am-v2n,1,Serializable,false,"Map(numRestoredFiles -> 0, removedFilesSize -> 21079, numRemovedFiles -> 1, restoredFilesSize -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 20464)",null,Databricks-Runtime/18.x-photon-scala2.13
1,2026-08-07T10:28:14.000Z,76835418810984,jack.gibb@inov8consulting.co.uk,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(431825730178474),c4deedfb-fb74-4993-9ad9-96530e48cd55,0807-080044-ic7c55am-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 815, numOutputBytes -> 21079)",null,Databricks-Runtime/18.x-photon-scala2.13
0,2026-08-07T10:28:12.000Z,76835418810984,jack.gibb@inov8consulting.co.uk,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(431825730178474),e267cde7-5b7e-49cd-a510-46c6ec46f876,0807-080044-ic7c55am-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 815, numOutputBytes -> 20464)",null,Databricks-Runtime/18.x-photon-scala2.13


In [0]:
# this shows us we can still access this customers record so it is not really deleted 
display(spark.sql(f"""SELECT * FROM {catalog}.{write_schema}.customer_version_history version as of 0 WHERE customer_id = 'CUST00377' """))

customer_id,first_name,last_name,date_of_birth,city,employment_status,annual_income,product_type,account_opened,credit_limit
CUST00377,Callum,Scott,08 Jul 1997,Aberdeen,Retired,52988,Credit Card,25 Jul 2016,1000


In [0]:
# Changing the retention time of the deleted files (this should not be done in a production environment this is purely for this exercise)
spark.sql(
  f"""
  ALTER TABLE {catalog}.{write_schema}.customer_version_history
  SET TBLPROPERTIES (
    delta.deletedFileRetentionDuration = 'interval 0 hours'
  )
  """
)

DataFrame[]

In [0]:
# Running the VACUUM Command to remove the file in question
spark.sql(f"""VACUUM {catalog}.{write_schema}.customer_version_history""")

DataFrame[path: string]

In [0]:
# Showing that we can no longer access this customers record
display(spark.sql(f"""SELECT * FROM {catalog}.{write_schema}.customer_version_history version as of 0 WHERE customer_id = 'CUST00377' """))

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5280427403369464>, line 1
----> 1 display(spark.sql(f"""SELECT * FROM {catalog}.{write_schema}.customer_version_history version as of 0 WHERE customer_id = 'CUST00377' """))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:901, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    898         _views.append(SubqueryAlias(df._plan, name))
    900 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 901 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    902 if "sql_command_result" in properties:
    903     df = DataFrame(CachedRelation(properties["sql_command_result"]), self)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)